# 06. MAF で AI Search による検索を行うエージェントを構築する

## コードの解説

Azure AI Search のインデックスに対して全文検索を行うカスタムツールを作成し、エージェントに組み込みます。
これは **RAG (Retrieval Augmented Generation)** パターンの基本的な実装です。
エージェントは検索ツールで関連ドキュメントを取得し、その情報をもとに回答を生成します。

### RAG の仕組み

```
ユーザー質問
    ↓
エージェントが search_documents ツールを呼び出す
    ↓
Azure AI Search でベクトル/全文検索
    ↓
関連ドキュメントを取得
    ↓
エージェントがドキュメントをもとに回答を生成
```

### 使用する Azure サービス

| サービス | 役割 |
|--------|------|
| Azure OpenAI | LLM による推論・応答生成 |
| Azure AI Search | ドキュメントのインデックス管理と検索 |

## Azure の操作手順

### 1. Azure AI Search リソースの作成

1. [Azure Portal](https://portal.azure.com) にアクセスし、「リソースの作成」をクリック
2. 「AI Search」を検索してリソースを作成
3. 価格レベルは **Free** または **Basic** を選択（ワークショップ用途）
4. 作成完了後、「キー」メニューから **管理者キー** をコピー
5. 「概要」ページから **エンドポイント URL** をコピー

### 2. データのアップロードとインデックスの作成

1. `data/` 配下にあるサンプルデータを Blob Storage にアップロード
2. AI Search リソースの「データのインポート」メニューを選択
3. Blob Storage をデータソースとして選択し、インデックスの作成
4. **インデックス名**をコピー

### 3. .env ファイルへの設定追加

```env
AZURE_SEARCH_ENDPOINT=https://<your-service-name>.search.windows.net
AZURE_SEARCH_API_KEY=<your-admin-key>
AZURE_SEARCH_INDEX_NAME=<your-index-name>
```

In [ ]:
import os
from typing import Annotated
from pydantic import Field
from dotenv import load_dotenv
from azure.search.documents import SearchClient
from azure.core.credentials import AzureKeyCredential
from agent_framework import Agent
from agent_framework.openai import OpenAIChatCompletionClient

load_dotenv()

AZURE_OPENAI_ENDPOINT   = os.getenv("AZURE_OPENAI_ENDPOINT")
AZURE_OPENAI_API_KEY    = os.getenv("AZURE_OPENAI_API_KEY")
MODEL_DEPLOYMENT_NAME   = os.getenv("AZURE_OPENAI_CHAT_DEPLOYMENT")
AZURE_SEARCH_ENDPOINT   = os.getenv("AZURE_SEARCH_ENDPOINT")
AZURE_SEARCH_API_KEY    = os.getenv("AZURE_SEARCH_API_KEY")
AZURE_SEARCH_INDEX_NAME = os.getenv("AZURE_SEARCH_INDEX_NAME")

USE_KEY_AUTH = os.getenv("USE_KEY_AUTH", "False").lower() in ("true", "1", "t")
if USE_KEY_AUTH:
    auth_kwargs = dict(api_key=AZURE_OPENAI_API_KEY)
else:
    from azure.identity.aio import DefaultAzureCredential
    os.environ.pop("AZURE_OPENAI_API_KEY", None)
    auth_kwargs = dict(credential=DefaultAzureCredential())


# ---------------------------------------------------------------
# Azure AI Search を呼び出すカスタムツールの定義
#
# azure-search-documents SDK を使って AI Search にアクセスする
# ポイント1: SearchClient は endpoint, index_name, credential で初期化
# ポイント2: search() メソッドで全文検索を実行、top で件数を制限
# ポイント3: ツールは str を返す（エージェントが読みやすい形式に整形）
# ---------------------------------------------------------------
def search_documents(
    query: Annotated[str, Field(description="検索クエリ文字列")],
    top: Annotated[int, Field(description="取得する最大件数（デフォルト: 3）")] = 3,
) -> str:
    """Azure AI Search でドキュメントを検索し、関連する内容を返します。"""
    search_client = SearchClient(
        endpoint=AZURE_SEARCH_ENDPOINT,
        index_name=AZURE_SEARCH_INDEX_NAME,
        credential=AzureKeyCredential(AZURE_SEARCH_API_KEY),
    )
    results = list(search_client.search(query, top=top))
    if not results:
        return "検索結果が見つかりませんでした。"
    docs = []
    for i, r in enumerate(results, 1):
        content = r.get("chunk") or str(r)
        docs.append(f"[{i}] {content[:400]}")
    return "\n".join(docs)


# ---------------------------------------------------------------
# AI Search ツール付きエージェントの作成
# ---------------------------------------------------------------
agent = Agent(
    client=OpenAIChatCompletionClient(
        **auth_kwargs,
        azure_endpoint=AZURE_OPENAI_ENDPOINT,
        model=MODEL_DEPLOYMENT_NAME,
    ),
    name="SearchAgent",
    instructions=(
        "あなたは架空装置に関するドキュメント検索エージェントです。"
        "QS7000・NP200・PF3X・HT9Z・TV500・CS100・ML2000・VSX3・PA888・SM410 などの装置に関する"
        "仕様書・操作マニュアル・安全ガイドライン・保守マニュアル・点検報告書・インストールマニュアル・"
        "性能試験報告書・品質管理・開発提案書・トラブルシューティングガイドを参照できます。"
        "search_documents ツールを使って関連情報を検索し、正確な回答を提供してください。"
        "検索結果に基づいた回答であることを明示してください。"
    ),
    tools=[search_documents],
)

# エージェントの実行
response = await agent.run("SM410 のトラブルシューティング方法を教えてください。")
print(response)
